# AML - SAIGE-GENE SINGLE VARIANT - ASSOCIATION ANALYSIS 
This script runs the single-variant association test for SAIGE-GENE. Logistic Firth regression is used with a Firth cutoff of 0.1 (the default setting). Imputation best guess from SAIGE-GENE was used for the single variant association analysis. The option is_output_moreDetails must be TRUE must be specified to receive the full report for the single-variant SAIGE-GENE+ results.

In [ ]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 
import pandas as pd

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


# COMMAND 

In [ ]:
%%writefile acaf_single_variant_deprivation_index_analysis.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # === ACAF SINGLE VARIANT ===
    for pheno in aml_phenotype; do 
    echo "Processing ancestry: ${pop} | phenotype: ${pheno} | chr: ${chrom}"
    
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_chr${chrom}_deprivation_index" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --idstoIncludeFile="${snplist}/${pop}_acaf_chr${chrom}_variant_qc.snplist" \
        --minMAF=0.01 \
        --impute_method=best_guess \
        --is_Firth_beta=TRUE \
        --pCutoffforFirth=0.1 \
        --GMMATmodelFile="${null_glmm}/single_variant_${pop}_${pheno}_deprivation_index_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/single_variant_${pop}_${pheno}_deprivation_index_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --is_fastTest=TRUE \
        --is_output_moreDetails=TRUE
                
    done
done


In [ ]:
# Copy script to workspace bucket 
!gsutil cp acaf_single_variant_deprivation_index_analysis.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh

# DSUB COMMAND 

In [ ]:
%%bash --out saige_gene_step02

LOWER=1
UPPER=2
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 880 \
  --min-ram 6.5 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=2
UPPER=3
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 930 \
  --min-ram 6.5 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=3
UPPER=4
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 730 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=4
UPPER=5
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 740 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=5
UPPER=6
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 680 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=6
UPPER=7
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 660 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=7
UPPER=8
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 640 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=8
UPPER=9
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 600 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=9
UPPER=10
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 490 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=10
UPPER=11
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 550 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=11
UPPER=12
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 520 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=12
UPPER=13
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 520 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=13
UPPER=14
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 380 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=14
UPPER=15
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 360 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=15
UPPER=16
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 350 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=16
UPPER=17
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 390 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=17
UPPER=18
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 360 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=18
UPPER=19
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 310 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=19
UPPER=20
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 300 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=20
UPPER=21
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 270 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=21
UPPER=22
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 180 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=22
UPPER=23
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 200 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}.fam" \
  --input-recursive snplist="${WORKSPACE_BUCKET}/data/wgs_qc/wgs_variant_qc/acaf_threshold/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/acaf_single_variant_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/acaf_single_variant_deprivation_index_analysis.sh"

done

## DOWNLOAD RESULTS FROM THE WORKSPACE BUCKET 

In [ ]:
# Remove directories 
!rm -rf saige_gene_step02
!rm -rf merged_saige_gene_step02  

In [ ]:
# Create new directories to ensure clean environment for download 
!mkdir -p saige_gene_step02
!mkdir -p merged_saige_gene_step02

In [ ]:
# Copy single variant summary statistics to directory 
!gsutil cp {my_bucket}/data/saige_step02/acaf_single_variant_analysis/*deprivation_index saige_gene_step02

In [ ]:
%%bash 
# Merge chromosome split files together into single file 
# Format the file into PLINK logistic regression output 
unmerge_files=saige_gene_step02
merge_files=merged_saige_gene_step02
for pheno in aml_phenotype; do  
    # Loop through all chromosomes 
    for chrom in {1..22}; do 
         # PLINK logistic regression header  
        echo -e "#CHROM\tPOS\tID\tREF\tALT\tA1\tTEST\tOBS_CT\tBETA\tSE\tT_STAT\tP" > $merge_files/EUR_${pheno}_plink_chr${chrom}_deprivation_index
        # Print chromosome in loop 
        echo "Processing phenotype: ${pheno} | chr: ${chrom}"
        # Format file into PLINK logistic regression file
        awk -v add="ADD" -v chr="$chrom" 'BEGIN {OFS="\t"} \
        NR > 1 {print chr, $2, $3, $4, $5, $5, add, ($5+$6), $9, $10, $11, $13}' \
        $unmerge_files/EUR_${pheno}_chr${chrom}_deprivation_index >> $merge_files/EUR_${pheno}_plink_chr${chrom}_deprivation_index
    done
done


In [ ]:
%%bash 
# Merge chromosome split files together into single file 
# Format the file into PLINK logistic regression output 
unmerge_files=saige_gene_step02
merge_files=merged_saige_gene_step02
for pheno in aml_phenotype; do 
    # PLINK logistic regression header  
    echo -e "#CHROM\tPOS\tID\tREF\tALT\tA1\tTEST\tOBS_CT\tBETA\tSE\tT_STAT\tP" > $merge_files/EUR_${pheno}_plink_deprivation_index
    # Loop through all chromosomes 
    for chrom in {1..22}; do 
        # Print chromosome in loop 
        echo "Processing phenotype: ${pheno} | chr: ${chrom}"
        # Format file into PLINK logistic regression file
        awk -v add="ADD" -v chr="$chrom" 'BEGIN {OFS="\t"} \
        NR > 1 {print chr, $2, $3, $4, $5, $5, add, ($5+$6), $9, $10, $11, $13}' \
        $unmerge_files/EUR_${pheno}_chr${chrom}_deprivation_index >> $merge_files/EUR_${pheno}_plink_deprivation_index
    done
done
